# Quiz 8

### Imports

In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from scipy.spatial.distance import mahalanobis

### Datasets

In [12]:
df_1 = pd.read_csv('datasets/homework_8.1.csv')
df_2 = pd.read_csv('datasets/homework_8.2.csv')

### Q1

In [9]:
# Predict X from Z
model = LogisticRegression()
model.fit(df_1[['Z']], df_1[['X']])

df_1['propensity'] = model.predict_proba(df_1[['Z']])[:, 1]
df_1['ipw'] = df_1['X'] / df_1['propensity'] + (1 - df_1['X']) / (1 - df_1['propensity'])

df_1.head()

/Users/ethanmarlatt/Desktop/OMDS/DX702/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,Unnamed: 0,X,Y,Z,propensity,ipw
0,0,1,4.109218,1.764052,0.840114,1.190315
1,1,0,2.259504,0.400157,0.584646,2.407585
2,2,0,-0.647584,0.978738,0.711082,3.461195
3,3,0,2.106071,2.240893,0.892793,9.327719
4,4,1,3.583464,1.867558,0.853089,1.172211


In [10]:
mu1 = (df_1['X'] * df_1['Y'] * df_1['ipw']).sum() / (df_1['X'] * df_1['ipw']).sum()
mu0 = ((1 - df_1['X']) * df_1['Y'] * df_1['ipw']).sum() / ((1 - df_1['X']) * df_1['ipw']).sum()

ate = mu1 - mu0
print(ate)

2.2743411898510137


### Q3

In [17]:
treated = df_2[df_2['X'] == 1].copy()
untreated = df_2[df_2['X'] == 0].copy()

Z = df_2[['Z1', 'Z2']].values.T

cov = np.cov(Z)
inv_cov = np.linalg.inv(cov)

X_treated = treated[['Z1', 'Z2']].values
X_untreated = untreated[['Z1', 'Z2']].values

n_treated = len(X_treated)
n_untreated = len(X_untreated)

dists = np.zeros((n_treated, n_untreated))
for i in range(n_treated):
    for j in range(n_untreated):
        dists[i, j] = mahalanobis(X_treated[i], X_untreated[j], VI=inv_cov)

matched_untreated_idx = dists.argmin(axis=1)
Y_control_for_treated = untreated['Y'].iloc[matched_untreated_idx].values
att = np.mean(treated['Y'].values - Y_control_for_treated)

matched_treated_idx = dists.argmin(axis=0)
Y_treated_for_control = treated['Y'].iloc[matched_treated_idx].values
atc = np.mean(Y_treated_for_control - untreated['Y'].values)

n_treated = len(treated)
n_untreated = len(untreated)
n_total = n_treated + n_untreated

ate = (n_treated * att + n_untreated * atc) / n_total

print(f"ATE: {ate}")

ATE: 3.4070474053689317


### Q4

In [25]:
min_dists_per_treated = dists.min(axis=1)

worst_treated_idx = min_dists_per_treated.argmax()
worst_treated_unit = treated.iloc[worst_treated_idx]

nearest_untreated_idx = dists[worst_treated_idx].argmin()
nearest_untreated_unit = untreated.iloc[nearest_untreated_idx]

print(f"Z1: {nearest_untreated_unit['Z1']:.4f}")
print(f"Z2: {nearest_untreated_unit['Z2']:.4f}")

Z1: 1.5200
Z2: -1.2822
